In [ ]:
from BASIS.modules import vis, imutils, likelihood
from BASIS.modules import optimisation as opt
from BASIS.models import base
import numpy as np
import matplotlib.pyplot as plt
import ehtim as eh
import nevergrad as ng
import os

In [ ]:
# Optimising to an image

fov = 225 # Field of view (uas)
imgdim = 64 # Image dimension (pixels)

model_choice = ['mgring3', 'gauss', 'gauss']  # Model choice for the image fitting

# Initialise objective image
objective_img = '../data/fits/s_sgra.fits'
objective_img = eh.image.load_fits(objective_img)
objective_img = objective_img.regrid_image(fov*1e-6/206265, imgdim, interp='cubic')

# Create and run optimiser
optimiser = opt.ModelOptimisation(model_list=model_choice, dim=imgdim, fov=fov)
print(f"Number of free parameters: {optimiser.ng_parameterisation.dimension}")
recommended_params = optimiser.optimise_img(objective_img, budget=1000, num_workers=8)

# Plot
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(objective_img._imdict['I'].reshape(imgdim, imgdim), extent=[-fov/2, fov/2, -fov/2, fov/2], origin='upper', cmap='inferno')
axs[1].imshow(optimiser.model.sky_map(recommended_params.value[0]).reshape(imgdim, imgdim), extent=[-fov/2, fov/2, -fov/2, fov/2], origin='upper', cmap='inferno')


In [ ]:
# Optimising to a dataset

fov = 225 # Field of view (uas)
imgdim = 64 # Image dimension (pixels)

model_choice = ['xsringauss', 'gauss']  # Model choice for the image fitting

# Initialise likelihood object
obs = eh.obsdata.load_uvfits('../data/uvfits/SR1_M87_2021_108_hilo_hops_netcal_StokesI.uvfits')
noise_factor = 1
static_noise_floor = 0.0001
noise_frac = 0.05
custom_likelihood = likelihood.ModelLikelihood(model_names=model_choice, obs=obs, count='min', imgdim=imgdim, fov=fov, dterms={'ci':100}, 
                                               noise_factor=noise_factor, static_noise_floor=static_noise_floor, noise_frac=noise_frac)

# Create and run optimiser
optimiser = opt.ModelOptimisation(model_list=model_choice, dim=imgdim, fov=fov)
print(f"Number of free parameters: {optimiser.ng_parameterisation.dimension}")
recommended_params = optimiser.optimise_data(custom_likelihood, budget=1000, num_workers=8)

_, _, model_data = custom_likelihood.sample_data(dict(zip(optimiser.model.params.keys(), recommended_params.value[0])), dtype='vis')

# normalising vis (optional)
custom_likelihood.obs.data['vis'] = custom_likelihood.obs.data['vis'] / np.max(np.abs(custom_likelihood.obs.data['vis'])) * np.max(np.abs(model_data.detach().cpu().numpy()))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(optimiser.model.sky_map(recommended_params.value[0]).reshape(imgdim, imgdim), extent=[-fov/2, fov/2, -fov/2, fov/2], origin='upper', cmap='inferno', interpolation='gaussian')
chi2 = custom_likelihood.reduced_chi2(dict(zip(optimiser.model.params.keys(), recommended_params.value[0])), dtype='ci')
ax.text(0.05, 0.95, f'Reduced Chi2: {chi2:.3f}', transform=ax.transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
custom_likelihood.plot_all(dict(zip(optimiser.model.params.keys(), recommended_params.value[0])))

